# Olist E-commerce: Growth and Rankings with Window Functions

**Questions:** How fast is revenue growing month over month and year over year?
Which categories lead, overall and in each state?

In [1]:
import duckdb

%load_ext sql
con = duckdb.connect("../data/olist.duckdb", read_only=True)
%sql con
%config SqlMagic.displaylimit = 50

In [2]:
%%sql
WITH monthly AS (
    SELECT date_trunc('month', o.order_purchase_timestamp)::DATE AS month,
           ROUND(SUM(i.price), 0) AS revenue
    FROM orders o
    JOIN order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp <  '2018-09-01'
    GROUP BY month
)
SELECT month,
       revenue,
       LAG(revenue) OVER (ORDER BY month) AS prev_month_revenue,
       ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month))
             / LAG(revenue) OVER (ORDER BY month), 1) AS mom_growth_pct
FROM monthly
ORDER BY month;

Running query in 'DuckDBPyConnection'

month,revenue,prev_month_revenue,mom_growth_pct
2017-01-01,111798.0,None,None
2017-02-01,234223.0,111798.0,109.5
2017-03-01,359199.0,234223.0,53.4
2017-04-01,340670.0,359199.0,-5.2
2017-05-01,489338.0,340670.0,43.6
2017-06-01,421923.0,489338.0,-13.8
2017-07-01,481605.0,421923.0,14.1
2017-08-01,554700.0,481605.0,15.2
2017-09-01,607400.0,554700.0,9.5
2017-10-01,648248.0,607400.0,6.7


In [3]:
%%sql
WITH monthly AS (
    SELECT date_trunc('month', o.order_purchase_timestamp)::DATE AS month,
           ROUND(SUM(i.price), 0) AS revenue
    FROM orders o
    JOIN order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp <  '2018-09-01'
    GROUP BY month
)
SELECT month,
       revenue,
       SUM(revenue) OVER (ORDER BY month) AS running_total,
       ROUND(100.0 * SUM(revenue) OVER (ORDER BY month) / SUM(revenue) OVER (), 1) AS pct_of_total,
       ROUND(AVG(revenue) OVER (ORDER BY month
                                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 0) AS moving_avg_3m
FROM monthly
ORDER BY month;

Running query in 'DuckDBPyConnection'

month,revenue,running_total,pct_of_total,moving_avg_3m
2017-01-01,111798.0,111798.0,0.8,111798.0
2017-02-01,234223.0,346021.0,2.6,173011.0
2017-03-01,359199.0,705220.0,5.4,235073.0
2017-04-01,340670.0,1045890.0,7.9,311364.0
2017-05-01,489338.0,1535228.0,11.6,396402.0
2017-06-01,421923.0,1957151.0,14.8,417310.0
2017-07-01,481605.0,2438756.0,18.5,464289.0
2017-08-01,554700.0,2993456.0,22.7,486076.0
2017-09-01,607400.0,3600856.0,27.3,547902.0
2017-10-01,648248.0,4249104.0,32.2,603449.0


In [4]:
%%sql
WITH monthly AS (
    SELECT date_trunc('month', o.order_purchase_timestamp)::DATE AS month,
           ROUND(SUM(i.price), 0) AS revenue
    FROM orders o
    JOIN order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp <  '2018-09-01'
    GROUP BY month
)
SELECT month,
       revenue,
       LAG(revenue, 12) OVER (ORDER BY month) AS same_month_last_year,
       ROUND(100.0 * (revenue - LAG(revenue, 12) OVER (ORDER BY month))
             / LAG(revenue, 12) OVER (ORDER BY month), 1) AS yoy_growth_pct
FROM monthly
ORDER BY month;

Running query in 'DuckDBPyConnection'

month,revenue,same_month_last_year,yoy_growth_pct
2017-01-01,111798.0,None,None
2017-02-01,234223.0,None,None
2017-03-01,359199.0,None,None
2017-04-01,340670.0,None,None
2017-05-01,489338.0,None,None
2017-06-01,421923.0,None,None
2017-07-01,481605.0,None,None
2017-08-01,554700.0,None,None
2017-09-01,607400.0,None,None
2017-10-01,648248.0,None,None


In [5]:
%%sql
WITH category_revenue AS (
    SELECT COALESCE(t.product_category_name_english, p.product_category_name, 'unknown') AS category,
           ROUND(SUM(i.price), 0) AS revenue
    FROM orders o
    JOIN order_items i ON o.order_id = i.order_id
    LEFT JOIN products p ON i.product_id = p.product_id
    LEFT JOIN category_translation t ON p.product_category_name = t.product_category_name
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp <  '2018-09-01'
    GROUP BY category
)
SELECT category,
       revenue,
       RANK() OVER (ORDER BY revenue DESC) AS revenue_rank,
       ROUND(100.0 * revenue / SUM(revenue) OVER (), 1) AS pct_of_revenue,
       ROUND(100.0 * SUM(revenue) OVER (ORDER BY revenue DESC) / SUM(revenue) OVER (), 1) AS cumulative_pct
FROM category_revenue
ORDER BY revenue_rank;

Running query in 'DuckDBPyConnection'

category,revenue,revenue_rank,pct_of_revenue,cumulative_pct
health_beauty,1229557.0,1,9.3,9.3
watches_gifts,1163466.0,2,8.8,18.2
bed_bath_table,1022956.0,3,7.8,25.9
sports_leisure,952840.0,4,7.2,33.1
computers_accessories,888056.0,5,6.7,39.9
furniture_decor,706237.0,6,5.4,45.2
housewares,614342.0,7,4.7,49.9
cool_stuff,609158.0,8,4.6,54.5
auto,577838.0,9,4.4,58.9
garden_tools,469135.0,10,3.6,62.5


In [6]:
%%sql
SELECT revenue,
       RANK()       OVER (ORDER BY revenue DESC) AS rank,
       DENSE_RANK() OVER (ORDER BY revenue DESC) AS dense_rank,
       ROW_NUMBER() OVER (ORDER BY revenue DESC) AS row_number
FROM (VALUES (100), (90), (90), (80)) AS t(revenue);

Running query in 'DuckDBPyConnection'

revenue,rank,dense_rank,row_number
100,1,1,1
90,2,2,2
90,2,2,3
80,4,3,4


In [7]:
%%sql
WITH state_category AS (
    SELECT c.customer_state AS state,
           COALESCE(t.product_category_name_english, p.product_category_name, 'unknown') AS category,
           ROUND(SUM(i.price), 0) AS revenue
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN order_items i ON o.order_id = i.order_id
    LEFT JOIN products p ON i.product_id = p.product_id
    LEFT JOIN category_translation t ON p.product_category_name = t.product_category_name
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp <  '2018-09-01'
    GROUP BY state, category
),
ranked AS (
    SELECT state, category, revenue,
           RANK() OVER (PARTITION BY state ORDER BY revenue DESC) AS rank_in_state
    FROM state_category
)
SELECT *
FROM ranked
WHERE state IN ('SP', 'RJ', 'MG')
  AND rank_in_state <= 3
ORDER BY state, rank_in_state;

Running query in 'DuckDBPyConnection'

state,category,revenue,rank_in_state
MG,health_beauty,153690.0,1
MG,bed_bath_table,128591.0,2
MG,watches_gifts,121622.0,3
RJ,watches_gifts,173496.0,1
RJ,bed_bath_table,144192.0,2
RJ,health_beauty,139538.0,3
SP,bed_bath_table,472208.0,1
SP,health_beauty,453027.0,2
SP,watches_gifts,420884.0,3


In [8]:
con.close()

## Findings: growth and rankings

- **Month over month:** fast growth through 2017 (Black Friday Nov 2017: +52%, then −27% in December
  as sales returned to normal). From March 2018, MoM growth is near zero.
- **Moving average (3 months)** peaked in **May 2018 at R$ 968K** and has slowly declined since,
  so growth has stopped.
- **Year over year:** every 2018 month is still above 2017, but the YoY rate fell from +727% (Jan)
  to **+51% (Aug)**. Both views are needed: "up 51% vs last year" and "flat for 6 months."
- **Pareto:** **18 of 74 categories (24%) make 80% of revenue.**
- **Regional differences:** health & beauty is #1 nationally and in 14 states, but São Paulo's #1
  is bed/bath/table and Rio's is watches & gifts. Campaigns should be regional.

## SQL lessons
- Window functions keep every row (GROUP BY squashes them). `OVER (PARTITION BY … ORDER BY …)` defines the window.
- `LAG` for growth, `SUM() OVER (ORDER BY)` for running totals, `ROWS BETWEEN` for moving averages.
- Window functions are calculated after WHERE, so filtering by rank needs a CTE.